# Analyse TAL des chronologies d’événements sportifs

## Objectif du projet

L’objectif de cette partie est d’appliquer le Traitement Automatique du Langage naturel (TAL) à des données issues des réseaux sociaux autour d’un match de football.

Nous cherchons à transformer des textes bruts, comme des titres de posts ou des commentaires, en informations structurées :

- type d’événement : match thread, fin de match, VAR, penalty, but ;
- score détecté ;
- moment temporel par rapport au match ;
- construction d’une timeline sociale.

Cette approche permet ensuite de comparer la chronologie sociale avec la chronologie officielle du match.

Nous importons ici les bibliothèques nécessaires :
- json pour lire le fichier de données ;
- re pour utiliser les expressions régulières ;
- pandas pour manipuler les données sous forme de tableau ;
- datetime pour calculer les minutes relatives au match.

In [12]:
import json
import re
import csv
import pandas as pd
from datetime import datetime

In [18]:
nom_fichier = "period_debug_galata_liverpool_20260310_1779694004.json"

with open(nom_fichier, "r", encoding="utf-8") as fichier:
    data = json.load(fichier)

print("Fichier chargé avec succès.")
print(data.keys())

posts = data["posts"]

print("Nombre total de posts :", len(posts))

Fichier chargé avec succès.
dict_keys(['match_id', 'generated_at_utc', 'config', 'summary', 'posts', 'matched_comments'])
Nombre total de posts : 200


# Partie 2 — Exploration du résumé du dataset

Avant d’appliquer le TAL, il est important de comprendre la taille et la qualité du corpus.

Le fichier contient un résumé statistique indiquant :
- le nombre de posts récupérés ;
- le nombre de commentaires scannés ;
- le nombre de commentaires situés dans les fenêtres temporelles du match ;
- le taux de correspondance.

Cette étape permet d’évaluer le volume de données et le niveau de bruit présent dans le corpus.

In [14]:
summary = data["summary"]

print("Posts récupérés :", summary["posts_fetched"])
print("Commentaires scannés :", summary["comments_scanned"])
print("Commentaires dans les fenêtres du match :", summary["comments_in_period_windows"])
print("Taux de correspondance :", summary["match_rate_percent"], "%")

Posts récupérés : 200
Commentaires scannés : 46416
Commentaires dans les fenêtres du match : 1228
Taux de correspondance : 2.65 %


# Partie 3 — Analyse des périodes du match

Le fichier découpe le match en plusieurs périodes : 00:00, 15:00, 30:00, 45:00, 60:00, 75:00 et 90:00.

Pour chaque période, le système indique combien de commentaires ont été trouvés dans une fenêtre temporelle autour de cette minute.

Cette analyse permet d’observer les moments du match où l’activité sociale est la plus forte.

In [15]:
per_period_counts = summary["per_period_counts"]

for periode, nombre in per_period_counts.items():
    print(periode, ":", nombre)

00:00 : 920
15:00 : 388
30:00 : 257
45:00 : 162
60:00 : 106
75:00 : 90
90:00 : 71


# Partie 4 — Observation des posts collectés

Après l’analyse globale du dataset, nous observons quelques publications collectées.

Chaque post contient notamment :
- une date de création ;
- un titre ;
- un nombre de commentaires récupérés ;
- un nombre de commentaires situés dans les fenêtres temporelles du match.

Cette étape permet de comprendre la nature des textes qui seront ensuite traités par le système TAL.

In [19]:
for post in posts[:10]:
    print("Heure :", post["created_time"])
    print("Titre :", post["title"])
    print("Commentaires récupérés :", post["comments_fetched"])
    print("Commentaires dans période :", post.get("comments_in_period", 0))
    print("-" * 60)

Heure : 2025-08-29 15:19:49 UTC
Titre : [Pre-match Thread] Tottenham Hotspur vs AFC Bournemouth (30/08/25)
Commentaires récupérés : 86
Commentaires dans période : 0
------------------------------------------------------------
Heure : 2026-03-19 10:09:51 UTC
Titre : Looking at the sequence of events around Liverpool vs Galatasaray – did the conditions feel fully balanced?
Commentaires récupérés : 9
Commentaires dans période : 0
------------------------------------------------------------
Heure : 2020-12-01 14:40:08 UTC
Titre : Each national team's youngest player: Where are they now?
Commentaires récupérés : 169
Commentaires dans période : 0
------------------------------------------------------------
Heure : 2023-12-05 13:35:28 UTC
Titre : [Press Conference] Erik ten Hag on if media stories re Man Utd players are true: "No. Of course in every team there are always players who are not playing or playing less who are less happy. In some circumstances you need that, they have to wait for 

Le filtrage lexical permet de réduire le bruit initial du corpus.  
Dans notre cas, nous avons conservé les publications mentionnant explicitement Galatasaray ou Liverpool, car elles sont les plus susceptibles d’être liées au match étudié.

# Partie 5 — Filtrage des posts pertinents

Le dataset contient un match principal : Galatasaray vs Liverpool.  
Cependant, certaines publications récupérées peuvent concerner d’autres matchs ou d’autres équipes.

Cette étape consiste à réduire le bruit du corpus en conservant uniquement les posts qui mentionnent explicitement Galatasaray ou Liverpool.

Ce filtrage est important car les données issues des réseaux sociaux sont souvent bruitées et ne sont pas toujours parfaitement centrées sur l’événement étudié.

In [20]:
posts_pertinents = []

for post in posts:
    titre = post["title"].lower()
    
    if "galatasaray" in titre or "liverpool" in titre:
        posts_pertinents.append(post)

print("Nombre total de posts :", len(posts))
print("Nombre de posts pertinents :", len(posts_pertinents))

Nombre total de posts : 200
Nombre de posts pertinents : 90


# Partie 6 — Nettoyage TAL des titres

Avant de détecter les événements, il est nécessaire de nettoyer les textes.

Le nettoyage consiste à :
- mettre le texte en minuscules ;
- supprimer les caractères inutiles ;
- conserver les symboles utiles aux scores comme `-` et `:` ;
- réduire les répétitions de lettres.

Cette étape permet d’obtenir des textes plus homogènes et plus faciles à analyser automatiquement.

In [21]:
def nettoyer_texte(texte):
    texte = texte.lower()
    texte = re.sub(r"[^\w\s\-:]", "", texte)
    texte = re.sub(r"(.)\1{2,}", r"\1", texte)
    return texte

In [22]:
for post in posts_pertinents[:10]:
    titre_original = post["title"]
    titre_nettoye = nettoyer_texte(titre_original)

    print("Original :", titre_original)
    print("Nettoyé  :", titre_nettoye)
    print("-" * 60)

Original : Looking at the sequence of events around Liverpool vs Galatasaray – did the conditions feel fully balanced?
Nettoyé  : looking at the sequence of events around liverpool vs galatasaray  did the conditions feel fully balanced
------------------------------------------------------------
Original : 10 years ago today, Jamie Vardy’s volley vs Liverpool.
Nettoyé  : 10 years ago today jamie vardys volley vs liverpool
------------------------------------------------------------
Original : Match Thread: Manchester United vs Liverpool | Premier League | 03 May 14:30 UTC
Nettoyé  : match thread: manchester united vs liverpool  premier league  03 may 14:30 utc
------------------------------------------------------------
Original : Post Match Thread: Galatasaray 1 : Liverpool 0 [Champions League MD2]
Nettoyé  : post match thread: galatasaray 1 : liverpool 0 champions league md2
------------------------------------------------------------
Original : Post Match Thread: Liverpool 7 - 0 Man

# Partie 7 — Détection des événements par règles

Dans cette première approche, nous utilisons des règles lexicales pour détecter les événements dans les titres.

Le principe est simple : si certains mots-clés apparaissent dans un titre, alors nous associons ce titre à une catégorie d’événement.

Par exemple :
- "post match" indique souvent une fin de match ;
- "match thread" indique un fil de discussion pendant le match ;
- "var" indique une décision liée à l’arbitrage vidéo ;
- "penalty" indique un penalty ;
- "goal" ou "but" indique un but.

Cette approche est simple, interprétable et constitue une première base avant l’utilisation de méthodes plus avancées.

In [23]:
def detecter_evenement(titre):
    texte = nettoyer_texte(titre)

    if "post match" in texte or "post-match" in texte:
        return "FIN_MATCH"
    
    elif "match thread" in texte:
        return "MATCH_THREAD"
    
    elif "var" in texte:
        return "VAR"
    
    elif "penalty" in texte or "pens" in texte:
        return "PENALTY"
    
    elif "goal" in texte or "but" in texte:
        return "BUT"
    
    elif "red card" in texte or "carton rouge" in texte:
        return "CARTON_ROUGE"
    
    else:
        return "AUTRE"

In [24]:
for post in posts_pertinents[:20]:
    titre = post["title"]
    evenement = detecter_evenement(titre)

    print("Titre :", titre)
    print("Événement détecté :", evenement)
    print("-" * 60)

Titre : Looking at the sequence of events around Liverpool vs Galatasaray – did the conditions feel fully balanced?
Événement détecté : AUTRE
------------------------------------------------------------
Titre : 10 years ago today, Jamie Vardy’s volley vs Liverpool.
Événement détecté : VAR
------------------------------------------------------------
Titre : Match Thread: Manchester United vs Liverpool | Premier League | 03 May 14:30 UTC
Événement détecté : MATCH_THREAD
------------------------------------------------------------
Titre : Post Match Thread: Galatasaray 1 : Liverpool 0 [Champions League MD2]
Événement détecté : FIN_MATCH
------------------------------------------------------------
Titre : Post Match Thread: Liverpool 7 - 0 Manchester United | English Premier League
Événement détecté : FIN_MATCH
------------------------------------------------------------
Titre : Post Match Thread: Manchester United 3-2 Liverpool
Événement détecté : FIN_MATCH
-------------------------------

# Partie 8 — Extraction automatique du score

En plus du type d’événement, certains titres contiennent un score final ou partiel.

L’objectif de cette étape est d’extraire automatiquement les scores présents dans les titres, par exemple :
- 1-0
- 2-1
- 3:2

Pour cela, nous utilisons une expression régulière permettant de repérer une structure numérique de type nombre-tiret-nombre ou nombre-deux-points-nombre.

In [25]:
def extraire_score(titre):
    match = re.search(r"\b\d+\s*[-:]\s*\d+\b", titre)

    if match:
        return match.group()
    else:
        return None

In [26]:
for post in posts_pertinents[:30]:
    titre = post["title"]
    score = extraire_score(titre)

    if score is not None:
        print("Titre :", titre)
        print("Score détecté :", score)
        print("-" * 60)

Titre : Match Thread: Manchester United vs Liverpool | Premier League | 03 May 14:30 UTC
Score détecté : 14:30
------------------------------------------------------------
Titre : Post Match Thread: Liverpool 7 - 0 Manchester United | English Premier League
Score détecté : 7 - 0
------------------------------------------------------------
Titre : Post Match Thread: Manchester United 3-2 Liverpool
Score détecté : 3-2
------------------------------------------------------------
Titre : Post Match Thread: Galatasaray 1-0 Liverpool
Score détecté : 1-0
------------------------------------------------------------
Titre : Post Match Thread: Liverpool 1-2 Manchester United
Score détecté : 1-2
------------------------------------------------------------
Titre : Post-Match Thread: Watford 3 - 0 Liverpool [ English Premier League ]
Score détecté : 3 - 0
------------------------------------------------------------
Titre : Post Match Thread: Liverpool 2-0 Manchester City | English Premier League 24

# Partie 9 — Construction du tableau final des résultats TAL

Après le filtrage, le nettoyage, la détection d’événements et l’extraction du score, nous construisons un tableau structuré.

Chaque ligne du tableau correspond à un post pertinent et contient :
- l’heure de publication ;
- le titre original ;
- le type d’événement détecté ;
- le score extrait si disponible ;
- le nombre de commentaires récupérés ;
- le nombre de commentaires associés aux périodes du match.

Cette transformation est essentielle car elle permet de passer de données textuelles brutes à une représentation structurée exploitable pour l’analyse.

In [27]:
resultats = []

for post in posts_pertinents:
    titre = post["title"]
    heure = post["created_time"]
    commentaires = post["comments_fetched"]
    commentaires_periode = post.get("comments_in_period", 0)

    evenement = detecter_evenement(titre)
    score = extraire_score(titre)

    ligne = {
        "heure": heure,
        "titre": titre,
        "evenement": evenement,
        "score": score,
        "commentaires": commentaires,
        "commentaires_periode": commentaires_periode
    }

    resultats.append(ligne)

df_resultats = pd.DataFrame(resultats)

df_resultats.head(10)

,heure,titre,evenement,score,commentaires,commentaires_periode
0,2026-03-19 10:09:51 UTC,Looking at the sequence of events around Liver...,AUTRE,None,9,0
1,2026-02-02 12:48:25 UTC,"10 years ago today, Jamie Vardy’s volley vs Li...",VAR,None,338,0
2,2026-05-03 13:27:26 UTC,Match Thread: Manchester United vs Liverpool |...,MATCH_THREAD,14:30,484,0
3,2025-09-30 21:02:05 UTC,Post Match Thread: Galatasaray 1 : Liverpool 0...,FIN_MATCH,None,466,0
4,2023-03-05 18:20:47 UTC,Post Match Thread: Liverpool 7 - 0 Manchester ...,FIN_MATCH,7 - 0,376,0
5,2026-05-03 16:26:27 UTC,Post Match Thread: Manchester United 3-2 Liver...,FIN_MATCH,3-2,477,0
6,2026-04-25 13:55:04 UTC,[Match Thread] Liverpool FC vs Crystal Palace,MATCH_THREAD,None,456,0
7,2025-01-06 09:54:13 UTC,Anfield roof leaking during last night's Liver...,AUTRE,None,293,0
8,2026-03-10 19:40:49 UTC,Post Match Thread: Galatasaray 1-0 Liverpool,FIN_MATCH,1-0,468,412
9,2021-08-31 14:10:17 UTC,BBC reporter randomly asking if they remember ...,BUT,None,479,0


# Partie 10 — Analyse quantitative des événements détectés

Après avoir détecté les événements dans les titres des posts, nous comptons le nombre d’occurrences de chaque catégorie.

Cette étape permet d’observer la distribution des types d’événements dans le corpus :
- combien de fils de match ;
- combien de posts d’après-match ;
- combien de mentions de VAR ;
- combien de penalties ou buts détectés ;
- combien de posts non classés.

Cela permet d’évaluer le comportement de notre première approche par règles.

In [29]:
compte_evenements = df_resultats["evenement"].value_counts()

print(compte_evenements)

evenement
MATCH_THREAD    35
AUTRE           28
FIN_MATCH       21
BUT              3
VAR              2
CARTON_ROUGE     1
Name: count, dtype: int64


# Partie 11 — Construction d’une timeline sociale

Après la détection des événements, nous construisons une timeline sociale.

L’objectif est de replacer les événements détectés dans l’ordre temporel afin d’observer comment les discussions sociales évoluent autour du match.

Nous filtrons d’abord les posts classés comme AUTRE, car ils ne correspondent pas à un événement identifié par nos règles.

In [30]:
df_timeline = df_resultats[df_resultats["evenement"] != "AUTRE"].copy()

df_timeline = df_timeline.sort_values(by="heure")

df_timeline[["heure", "evenement", "score", "titre"]].head(30)

,heure,evenement,score,titre
51,2018-05-26 20:38:14 UTC,FIN_MATCH,3 - 1,[Post Match Thread] Real Madrid 3 - 1 Liverpoo...
33,2019-06-01 18:20:31 UTC,MATCH_THREAD,None,Match Thread: Tottenham vs. Liverpool [UEFA Ch...
17,2019-12-29 17:21:30 UTC,VAR,None,Liverpool Vs Wolves - VAR Rules out Neto's Equ...
12,2020-02-29 19:25:18 UTC,FIN_MATCH,3 - 0,Post-Match Thread: Watford 3 - 0 Liverpool [ E...
9,2021-08-31 14:10:17 UTC,BUT,None,BBC reporter randomly asking if they remember ...
39,2021-12-28 21:54:22 UTC,FIN_MATCH,1-0,Post Match Thread: Leicester City 1-0 Liverpoo...
30,2022-05-28 18:40:52 UTC,MATCH_THREAD,None,Match Thread: Liverpool vs Real Madrid | UEFA ...
16,2022-05-28 21:33:00 UTC,FIN_MATCH,None,Post Match Thread: Liverpool 0 vs 1 Real Madri...
4,2023-03-05 18:20:47 UTC,FIN_MATCH,7 - 0,Post Match Thread: Liverpool 7 - 0 Manchester ...
24,2023-03-05 18:20:47 UTC,FIN_MATCH,7 - 0,[Post Match Thread] Liverpool 7 - 0 Manchester...


# Partie 12 — Calcul de la minute relative au match

Pour comparer les événements sociaux avec la chronologie officielle du match, nous convertissons les dates de publication en minutes relatives au début du match.

Le fichier indique l’heure de début du match.  
À partir de cette information, nous calculons l’écart entre l’heure de publication de chaque post et l’heure de début.

Cela permet d’obtenir une timeline sportive plus interprétable :
- minute négative : avant le match ;
- minute entre 0 et 120 : pendant le match ;
- minute supérieure à 120 : après le match.

In [31]:
heure_debut_match = data["config"]["match_start_utc"]

print("Heure de début du match :", heure_debut_match)

Heure de début du match : 2026-03-10T20:00:00Z


In [32]:
def calculer_minute_match(heure_post, heure_debut_match):
    format_post = "%Y-%m-%d %H:%M:%S UTC"
    format_match = "%Y-%m-%dT%H:%M:%SZ"

    temps_post = datetime.strptime(heure_post, format_post)
    temps_match = datetime.strptime(heure_debut_match, format_match)

    difference = temps_post - temps_match

    minute = int(difference.total_seconds() // 60)

    return minute

In [33]:
df_resultats["minute_match"] = df_resultats["heure"].apply(
    lambda h: calculer_minute_match(h, heure_debut_match)
)

df_resultats[["heure", "minute_match", "evenement", "score", "titre"]].head(10)

,heure,minute_match,evenement,score,titre
0,2026-03-19 10:09:51 UTC,12369,AUTRE,None,Looking at the sequence of events around Liver...
1,2026-02-02 12:48:25 UTC,-52272,VAR,None,"10 years ago today, Jamie Vardy’s volley vs Li..."
2,2026-05-03 13:27:26 UTC,77367,MATCH_THREAD,14:30,Match Thread: Manchester United vs Liverpool |...
3,2025-09-30 21:02:05 UTC,-231778,FIN_MATCH,None,Post Match Thread: Galatasaray 1 : Liverpool 0...
4,2023-03-05 18:20:47 UTC,-1585540,FIN_MATCH,7 - 0,Post Match Thread: Liverpool 7 - 0 Manchester ...
5,2026-05-03 16:26:27 UTC,77546,FIN_MATCH,3-2,Post Match Thread: Manchester United 3-2 Liver...
6,2026-04-25 13:55:04 UTC,65875,MATCH_THREAD,None,[Match Thread] Liverpool FC vs Crystal Palace
7,2025-01-06 09:54:13 UTC,-616926,AUTRE,None,Anfield roof leaking during last night's Liver...
8,2026-03-10 19:40:49 UTC,-20,FIN_MATCH,1-0,Post Match Thread: Galatasaray 1-0 Liverpool
9,2021-08-31 14:10:17 UTC,-2379230,BUT,None,BBC reporter randomly asking if they remember ...


In [35]:
df_timeline = df_resultats[df_resultats["evenement"] != "AUTRE"].copy()
df_timeline = df_timeline.sort_values(by="minute_match")

df_timeline[["minute_match", "heure", "evenement", "score", "titre"]].head(30)

,minute_match,heure,evenement,score,titre
51,-4096762,2018-05-26 20:38:14 UTC,FIN_MATCH,3 - 1,[Post Match Thread] Real Madrid 3 - 1 Liverpoo...
33,-3562660,2019-06-01 18:20:31 UTC,MATCH_THREAD,None,Match Thread: Tottenham vs. Liverpool [UEFA Ch...
17,-3258879,2019-12-29 17:21:30 UTC,VAR,None,Liverpool Vs Wolves - VAR Rules out Neto's Equ...
12,-3169475,2020-02-29 19:25:18 UTC,FIN_MATCH,3 - 0,Post-Match Thread: Watford 3 - 0 Liverpool [ E...
9,-2379230,2021-08-31 14:10:17 UTC,BUT,None,BBC reporter randomly asking if they remember ...
39,-2207406,2021-12-28 21:54:22 UTC,FIN_MATCH,1-0,Post Match Thread: Leicester City 1-0 Liverpoo...
30,-1990160,2022-05-28 18:40:52 UTC,MATCH_THREAD,None,Match Thread: Liverpool vs Real Madrid | UEFA ...
16,-1989987,2022-05-28 21:33:00 UTC,FIN_MATCH,None,Post Match Thread: Liverpool 0 vs 1 Real Madri...
4,-1585540,2023-03-05 18:20:47 UTC,FIN_MATCH,7 - 0,Post Match Thread: Liverpool 7 - 0 Manchester ...
24,-1585540,2023-03-05 18:20:47 UTC,FIN_MATCH,7 - 0,[Post Match Thread] Liverpool 7 - 0 Manchester...


# Partie 13 — Catégorisation temporelle des posts

Après avoir calculé la minute relative au match, nous classons chaque publication selon sa position temporelle.

Nous distinguons trois catégories :
- AVANT_MATCH : publications avant le coup d’envoi ;
- PENDANT_MATCH : publications pendant le match ;
- APRES_MATCH : publications après la fin estimée du match.

Cette catégorisation permet de mieux interpréter les discussions sociales selon le moment où elles apparaissent.

In [36]:
def categoriser_temps(minute_match):
    if minute_match < 0:
        return "AVANT_MATCH"
    elif minute_match <= 120:
        return "PENDANT_MATCH"
    else:
        return "APRES_MATCH"

In [37]:
df_resultats["categorie_temps"] = df_resultats["minute_match"].apply(categoriser_temps)

df_resultats[["minute_match", "categorie_temps", "evenement", "titre"]].head(20)

,minute_match,categorie_temps,evenement,titre
0,12369,APRES_MATCH,AUTRE,Looking at the sequence of events around Liver...
1,-52272,AVANT_MATCH,VAR,"10 years ago today, Jamie Vardy’s volley vs Li..."
2,77367,APRES_MATCH,MATCH_THREAD,Match Thread: Manchester United vs Liverpool |...
3,-231778,AVANT_MATCH,FIN_MATCH,Post Match Thread: Galatasaray 1 : Liverpool 0...
4,-1585540,AVANT_MATCH,FIN_MATCH,Post Match Thread: Liverpool 7 - 0 Manchester ...
5,77546,APRES_MATCH,FIN_MATCH,Post Match Thread: Manchester United 3-2 Liver...
6,65875,APRES_MATCH,MATCH_THREAD,[Match Thread] Liverpool FC vs Crystal Palace
7,-616926,AVANT_MATCH,AUTRE,Anfield roof leaking during last night's Liver...
8,-20,AVANT_MATCH,FIN_MATCH,Post Match Thread: Galatasaray 1-0 Liverpool
9,-2379230,AVANT_MATCH,BUT,BBC reporter randomly asking if they remember ...


In [38]:
df_timeline = df_resultats[df_resultats["evenement"] != "AUTRE"].copy()
df_timeline = df_timeline.sort_values(by="minute_match")

df_timeline[["minute_match", "categorie_temps", "evenement", "score", "titre"]].head(30)

,minute_match,categorie_temps,evenement,score,titre
51,-4096762,AVANT_MATCH,FIN_MATCH,3 - 1,[Post Match Thread] Real Madrid 3 - 1 Liverpoo...
33,-3562660,AVANT_MATCH,MATCH_THREAD,None,Match Thread: Tottenham vs. Liverpool [UEFA Ch...
17,-3258879,AVANT_MATCH,VAR,None,Liverpool Vs Wolves - VAR Rules out Neto's Equ...
12,-3169475,AVANT_MATCH,FIN_MATCH,3 - 0,Post-Match Thread: Watford 3 - 0 Liverpool [ E...
9,-2379230,AVANT_MATCH,BUT,None,BBC reporter randomly asking if they remember ...
39,-2207406,AVANT_MATCH,FIN_MATCH,1-0,Post Match Thread: Leicester City 1-0 Liverpoo...
30,-1990160,AVANT_MATCH,MATCH_THREAD,None,Match Thread: Liverpool vs Real Madrid | UEFA ...
16,-1989987,AVANT_MATCH,FIN_MATCH,None,Post Match Thread: Liverpool 0 vs 1 Real Madri...
4,-1585540,AVANT_MATCH,FIN_MATCH,7 - 0,Post Match Thread: Liverpool 7 - 0 Manchester ...
24,-1585540,AVANT_MATCH,FIN_MATCH,7 - 0,[Post Match Thread] Liverpool 7 - 0 Manchester...


# Partie 14 — Analyse croisée : événements et temporalité

Après avoir identifié les événements et catégorisé les publications selon le temps du match, nous analysons la répartition des événements dans chaque période temporelle.

Cette étape permet d’observer :
- quels types d’événements apparaissent avant le match ;
- quels événements sont détectés pendant le match ;
- quels contenus apparaissent après le match.

Cette analyse est utile pour comprendre la dynamique sociale autour de l’événement sportif.

In [39]:
table_evenements_temps = pd.crosstab(
    df_resultats["categorie_temps"],
    df_resultats["evenement"]
)

table_evenements_temps

evenement,AUTRE,BUT,CARTON_ROUGE,FIN_MATCH,MATCH_THREAD,VAR
categorie_temps,,,,,,
APRES_MATCH,6,0,0,2,17,0
AVANT_MATCH,21,3,1,19,18,2
PENDANT_MATCH,1,0,0,0,0,0


# Partie 15 — Export des résultats TAL

Après la détection des événements et l’enrichissement temporel, nous exportons les résultats dans un fichier CSV.

Cela permet de conserver une sortie structurée du pipeline TAL et de la réutiliser pour :
- la visualisation ;
- l’analyse statistique ;
- la comparaison avec une chronologie officielle ;
- la rédaction du rapport.

In [40]:
df_resultats.to_csv("resultats_tal_galata_liverpool.csv", index=False, encoding="utf-8")

print("Fichier exporté avec succès : resultats_tal_galata_liverpool.csv")

Fichier exporté avec succès : resultats_tal_galata_liverpool.csv


In [41]:
df_verification = pd.read_csv("resultats_tal_galata_liverpool.csv")

df_verification.head()

,heure,titre,evenement,score,commentaires,commentaires_periode,minute_match,categorie_temps
0,2026-03-19 10:09:51 UTC,Looking at the sequence of events around Liver...,AUTRE,NaN,9,0,12369,APRES_MATCH
1,2026-02-02 12:48:25 UTC,"10 years ago today, Jamie Vardy’s volley vs Li...",VAR,NaN,338,0,-52272,AVANT_MATCH
2,2026-05-03 13:27:26 UTC,Match Thread: Manchester United vs Liverpool |...,MATCH_THREAD,14:30,484,0,77367,APRES_MATCH
3,2025-09-30 21:02:05 UTC,Post Match Thread: Galatasaray 1 : Liverpool 0...,FIN_MATCH,NaN,466,0,-231778,AVANT_MATCH
4,2023-03-05 18:20:47 UTC,Post Match Thread: Liverpool 7 - 0 Manchester ...,FIN_MATCH,7 - 0,376,0,-1585540,AVANT_MATCH


# Partie 16 — Limites de l’approche par règles

L’approche par règles présente plusieurs avantages :
- elle est simple à comprendre ;
- elle est rapide à mettre en place ;
- elle est interprétable ;
- chaque décision peut être reliée à un mot-clé.

Cependant, elle présente aussi plusieurs limites.

Premièrement, elle dépend fortement des mots-clés choisis.  
Si un événement est formulé autrement, il peut ne pas être détecté.

Deuxièmement, elle détecte mal les événements implicites.  
Par exemple, une phrase comme "What a finish from Osimhen" peut évoquer un but, mais ne contient pas forcément les mots "goal" ou "but".

Troisièmement, elle est sensible au bruit des réseaux sociaux : abréviations, fautes, ironie, émotions, langage familier.

Enfin, cette approche ne comprend pas réellement le sens du texte.  
Elle vérifie seulement la présence de certains mots.

Ces limites justifient l’introduction ultérieure de méthodes plus avancées comme TF-IDF, la classification supervisée ou les modèles de langue.

In [42]:
df_autres = df_resultats[df_resultats["evenement"] == "AUTRE"]

df_autres[["heure", "titre"]].head(20)

,heure,titre
0,2026-03-19 10:09:51 UTC,Looking at the sequence of events around Liver...
7,2025-01-06 09:54:13 UTC,Anfield roof leaking during last night's Liver...
11,2024-08-29 11:29:40 UTC,Liverpool vs Man United aggregate score in the...
19,2025-02-07 07:46:32 UTC,A mascot joins Son for his warmup before the m...
50,2017-10-14 12:08:34 UTC,Amazing save by De Gea vs Liverpool
53,2026-02-27 11:17:53 UTC,We will face Galatasaray in the @ChampionsLeag...
54,2026-03-18 22:22:03 UTC,(@Squawka) Florian Wirtz created eight chances...
57,2025-09-30 21:00:47 UTC,[CHAMPIONS LEAGUE] Galatasaray 1-0 Liverpool
59,2026-01-28 21:58:50 UTC,"We will play one of Juventus, Atletico Madrid,..."
61,2026-03-10 19:39:14 UTC,Galatasaray once again defeats Liverpool at home!


In [43]:
total_posts = len(df_resultats)
total_autres = len(df_autres)

pourcentage_autres = (total_autres / total_posts) * 100

print("Nombre total de posts :", total_posts)
print("Nombre de posts classés AUTRE :", total_autres)
print("Pourcentage de AUTRE :", round(pourcentage_autres, 2), "%")

Nombre total de posts : 90
Nombre de posts classés AUTRE : 28
Pourcentage de AUTRE : 31.11 %


## Conclusion de l’approche par règles

Cette première approche permet de construire une base fonctionnelle pour l’extraction d’événements sportifs à partir de textes sociaux.

Elle permet de détecter des événements explicites comme les fils de match, les posts d’après-match, les penalties ou les mentions de VAR.

Cependant, elle reste limitée face aux formulations implicites, au bruit linguistique et à la diversité des expressions utilisées sur les réseaux sociaux.

Cette limite motive le passage à une approche plus avancée basée sur la vectorisation TF-IDF et la classification automatique.